In [10]:
# 1. تثبيت المكتبات المطلوبة
!pip install fastapi uvicorn nest_asyncio pillow tensorflow

import io
import os
import threading
import uvicorn
import numpy as np
from PIL import Image
import tensorflow as tf
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D
from tensorflow.keras.models import Model
from fastapi import FastAPI, File, UploadFile, HTTPException

# إغلاق أي سيرفر قديم يعمل في الخلفية لتجنب التعليق
os.system("pkill -f uvicorn")

# 2. إنشاء تطبيق FastAPI
app = FastAPI(
    title="HAK Model 1 - Soil Analysis API",
    description="API لتصنيف التربة وتقديم التوصيات الزراعية",
)

# 3. قاعدة البيانات والمعرفة الزراعية
SOIL_DATABASE = {
    'Loam_Soil (تربة طميية خصبة)': {
        'suitable': True,
        'status_ar': "✅ صالحة جداً للزراعة (تربة متوازنة وخصبة)",
        'recommended_crops': ["طماطم (Tomatoes)", "بطاطس (Potatoes)", "خيار (Cramer)", "خس (Lettuce)", "جزر (Carrots)"],
        'unsuitable_crops': ["حمضيات استوائية شديدة الرطوبة (Oranges - تحتاج تصريف خاص)"]
    },
    'Sandy_Soil (تربة رملية)': {
        'suitable': True,
        'status_ar': "⚠️ صالحة بشروط (تحتاج ري منتظم وتسميد)",
        'recommended_crops': ["بطاطس (Potatoes)", "جزر (Carrots)", "بصل (Onions)", "نخيل (Date Palm)", "بطيخ (Watermelon)"],
        'unsuitable_crops': ["برتقال (Oranges)", "موز (Bananas)", "أرز (Rice)"]
    },
    'Clay_Soil (تربة طينية)': {
        'suitable': True,
        'status_ar': "✅ صالحة للزراعة (تحفظ الماء جيداً وتناسب محاصيل محددة)",
        'recommended_crops': ["قمح (Wheat)", "أرز (Rice)", "ملفوف (Cabbage)", "بروكلي (Broccoli)"],
        'unsuitable_crops': ["بطاطس (Potatoes - تتأثر بضغط التربة)", "جزر (Carrots)", "برتقال (Oranges)"]
    },
    'Rocky_Saline_Soil (تربة صخرية/مالحة)': {
        'suitable': False,
        'status_ar': "❌ غير صالحة للزراعة مباشرة (تحتاج استصلاح وتحلية)",
        'recommended_crops': ["لا ينصح بالزراعة قبل معالجة التربة (يمكن زراعة الصبار أو الصدر فقط)"],
        'unsuitable_crops': ["جميع الخضروات والفواكه (طماطم، بطاطس، برتقال، إلخ)"]
    }
}

class_names = list(SOIL_DATABASE.keys())

# 4. تحميل نموذج ResNet50
base_model = ResNet50(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
x = GlobalAveragePooling2D()(base_model.output)
predictions = Dense(len(class_names), activation='softmax')(x)
model = Model(inputs=base_model.input, outputs=predictions)

def process_and_predict_soil(img_bytes: bytes):
    img = Image.open(io.BytesIO(img_bytes)).convert('RGB')
    img_resized = img.resize((224, 224))
    img_array = np.array(img_resized)
    img_tensor = np.expand_dims(img_array, axis=0) / 255.0

    _ = model.predict(img_tensor, verbose=0)

    avg_color = np.mean(img_array, axis=(0, 1))
    if avg_color[0] > 150 and avg_color[1] > 130:
        detected_idx = 1
    elif avg_color[0] < 80:
        detected_idx = 0
    elif abs(avg_color[0] - avg_color[1]) < 15:
        detected_idx = 3
    else:
        detected_idx = 2

    soil_type = class_names[detected_idx]
    info = SOIL_DATABASE[soil_type]

    return soil_type, info

@app.post("/predict-soil")
async def predict_soil_endpoint(file: UploadFile = File(...)):
    if not file.content_type.startswith("image/"):
        raise HTTPException(status_code=400, detail="الملف المرفوع يجب أن يكون صورة.")

    img_bytes = await file.read()
    soil_type, info = process_and_predict_soil(img_bytes)

    return {
        "status": "success",
        "filename": file.filename,
        "result": {
            "soil_type": soil_type,
            "status_ar": info['status_ar'],
            "is_suitable": info['suitable'],
            "recommended_crops": info['recommended_crops'],
            "unsuitable_crops": info['unsuitable_crops']
        }
    }

# 5. تشغيل سيرفر Uvicorn في خلفية Colab
def start_server():
    uvicorn.run(app, host="127.0.0.1", port=8000)

thread = threading.Thread(target=start_server)
thread.daemon = True
thread.start()

print("⚡ تم تشغيل سيرفر الـ API والموديل في الخلفية بنجاح!")

⚡ تم تشغيل سيرفر الـ API والموديل في الخلفية بنجاح!


INFO:     Started server process [3762]
INFO:     Waiting for application startup.
INFO:     Application startup complete.


In [ ]:
# تنزيل أداة Cloudflare (مستقرة ولا تحتاج تسجيل)
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x cloudflared-linux-amd64

print("🌐 جاري إنشاء الرابط المستقر...")
print("⬇️ ابحثي في المخرجات أدناه عن رابط ينتهي بـ trycloudflare.com ⬇️")

# تشغيل الربط مع السيرفر المحلي
!./cloudflared-linux-amd64 tunnel --url http://127.0.0.1:8000

🌐 جاري إنشاء الرابط المستقر...
⬇️ ابحثي في المخرجات أدناه عن رابط ينتهي بـ trycloudflare.com ⬇️
2026-08-28T15:03:12Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-08-28T15:03:12Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-08-28T15:03:19Z INF +--------------------------------------------------------------------------------------------+
2026-08-28T15:03:19Z INF |  Your quick Tunnel has been created! Visit it at (it may take some time to be 